# Exercise 0 - PydanticAI and OpenRouter 

0. Summarize job ads

0a) Read all the jobs ads into python - IMPORTERA DATA

Läs in alla jobbannonser i Python

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()

job_ads = []
for file in Path("data").glob("ads*.txt"):
    with open(file, "r", encoding="utf-8") as f:
        job_ads.append(f.read())

print(f"Loaded {len(job_ads)} job ads")
print(job_ads[0][:500])

Vad händer här?
- Loopar igenom alla filer
- Sparar både namn + innehåll
- Returnerar lista → lätt att använda senare

0b) Create a function that uses PydanticAI agent to summarize a job ad. This function should take in an ad as its parameter and return a summary. - CREATE AGENT

Läs in alla jobbannonser i Python

## Data models

In [ ]:
from pydantic import BaseModel

class JobsAdSummary(BaseModel):
    job_title: str
    description: str
    summary: str
    responsibilities: list[str]
    words_in_article: int

In [ ]:


print("KEY:", os.getenv("OPENROUTER_API_KEY"))

0e) Make the output structured using a pydantic model together with PydanticAI agent. The output from the agent should have the following fields:

- job_title
- description
- summary
- responsibilities
- words_in_article

Gör output STRUKTURERAD med Pydantic

## Create agent

In [ ]:
from pydantic_ai import Agent


job_ads_agent = Agent(
    "openrouter:nvidia/nemotron-nano-12b-v2-vl:free",
    system_prompt="""
    You summarize job ads clearly and concisely.
    
    Extract:
    - job_title
    - description
    - summary
    - responsibilities
    - words_in_article

    Rules: 
    - Only extract information supported by the ad.
    - Keep summary short and factual
    - Responsibilities should be a list of main responsibilities.
    - words_in_article should be the approximate number of words in the full input article.
    - use the same language as the text when appropriate.

    """
)

async def summarize_job_ad(ad: str) -> JobsAdSummary:
    result = await job_ads_agent.run(ad, output_type=JobsAdSummary)
    return result.output

summaries = []
for ad in job_ads:
    summary = await summarize_job_ad(ad)
    summaries.append(summary)

summaries


Viktigt
- Här är output fortfarande ostrukturerad text
- Vi fixar det i steg (e)

0c) Now create and export markdown files for each job ad and its corresponding summary.

Skapa och exportera Markdown-filer

In [6]:
output_dir = Path("markdown_exports")
output_dir.mkdir(exist_ok=True)

for i, (ad, summary) in enumerate(zip(job_ads, summaries), start=1):
    md_content = f"""# Job Ad {i}
Orginal ad
{ad}

Summary
Job title: {summary.job_title}
Description: {summary.description}
Job title: {summary.summary}
Responsibilities:
"""
    for responsibility in summary.responsibilities:
        md_content += f" -{responsibility}\n"

    md_content += f"\n- Words in article: {summary.words_in_article}\n"

    with open(output_dir / f"jobad{i}.md", "w", encoding="utf-8") as f:
        f.write(md_content)

0d) Try to take other job ads from arbetsförmedlingen.se and see how well your function performs.

Testa med nya jobbannonser